<a href="https://colab.research.google.com/github/DrewThomasson/ebook2audiobook/blob/main/Notebooks/colab_ebook2audiobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS
- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Hardware
ebook2audiobook detects the hardware of the **notebook runtime**, not of your own computer, and falls back to CPU when there is no GPU. Choose the accelerator in **Runtime → Change runtime type** before running the cell below.
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/DrewThomasson/ebook2audiobook)

In [ ]:
# @title 🚀 Run ebook2audiobook!

import os
import re
import subprocess
import time
from collections import deque
from pathlib import Path
from queue import Empty, Queue
from threading import Thread
from typing import Optional

from IPython.display import clear_output

REPO_DIR = Path("/content/ebook2audiobook")
REPO_URL = "https://github.com/DrewThomasson/ebook2audiobook.git"
REPO_BRANCH = "main"
LOG_PATH = Path("/content/ebook2audiobook_install.log")

def clone_repo(repo_dir:Path)->None:
    if (repo_dir / ".git").exists():
        print("✅ ebook2audiobook is already downloaded.", flush=True)
        return
    if repo_dir.exists():
        raise RuntimeError(
            f"{repo_dir} exists but is not a Git checkout. "
            "Restart the notebook session and try again."
        )
    print("📚 Getting ebook2audiobook ready...", flush=True)
    subprocess.run(
        ["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)],
        check=True,
    )

def build_env()->dict:
    # TMPDIR is deliberately not set: ebook2audiobook.command and lib/conf.py both
    # pin it to <repo>/run, so anything passed from here is overwritten anyway.
    # Nothing device related either: ebook2audiobook.command detects the hardware.
    env = os.environ.copy()
    env.update({
        "MPLBACKEND": "Agg",
        "PYTHONUNBUFFERED": "1",
    })
    return env

clone_repo(REPO_DIR)

progress_terms = (
    "creating ./python_env",
    "hardware detected",
    "installing the right library packages",
    "installing python",
    "installing missing",
    "all required packages",
    "downloaded",
    "running on public",
    "gradio.live",
)
error_terms = ("error", "failed", "traceback", "exception", "critical", "fatal")

server_started = False
public_url = None
status_history = deque(maxlen=12)
recent_log_lines = deque(maxlen=40)

def render_status(message:Optional[str]=None)->None:
    if message and (not status_history or status_history[-1] != message):
        status_history.append(message)
    clear_output(wait=True)
    print("🚀 ebook2audiobook\n")
    if public_url:
        print("🎉 Ready to use!")
        print(f"🌐 Open the app: {public_url}")
        print("📌 Keep this cell running while you use ebook2audiobook.\n")
    else:
        print("⏳ First-time setup usually takes 20–60 minutes.")
        print("💡 You can safely leave this tab open while setup finishes.\n")
    if status_history:
        print("Latest activity:")
        for status in status_history:
            print(f"  {status}")
    print(f"\n📝 Technical details: {LOG_PATH}")

def friendly_status(line:str)->Optional[str]:
    text = line.strip()
    lower = text.lower()
    if "creating ./python_env" in lower:
        return "🐍 Preparing the supported Python environment..."
    if "hardware detected" in lower:
        found = re.search(r"'tag':\s*'([^']+)'", text)
        return f"🎮 Hardware detected: {found.group(1)}" if found else "🎮 Hardware detected."
    if "installing the right library packages" in lower:
        return "📦 Installing device support packages..."
    if "installing python" in lower or "installing missing" in lower:
        return "📦 Installing ebook2audiobook packages..."
    if "all required packages" in lower:
        return "✅ All required packages are installed."
    if lower.startswith("downloaded "):
        return f"📥 {text}"
    if server_started and any(term in lower for term in error_terms):
        return f"⚠️ {text}"
    return None

render_status("📚 Starting setup...")

process = subprocess.Popen(
    ["bash", "./ebook2audiobook.command", "--share"],
    cwd=REPO_DIR,
    env=build_env(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_queue = Queue()

def read_output()->None:
    assert process.stdout is not None
    for output_line in process.stdout:
        output_queue.put(output_line)
    output_queue.put(None)

reader = Thread(target=read_output, daemon=True)
reader.start()

try:
    with LOG_PATH.open("w", encoding="utf-8") as log:
        while True:
            try:
                line = output_queue.get(timeout=30)
            except Empty:
                if not server_started:
                    render_status(f"⏳ Setup is still working... ({time.strftime('%H:%M:%S')})")
                continue

            if line is None:
                break

            log.write(line)
            log.flush()
            recent_log_lines.append(line)
            line_lower = line.lower()

            url_match = re.search(r'https://[^ ]+\.gradio\.live', line)
            if url_match:
                public_url = url_match.group(0)
                server_started = True
                render_status("✅ Setup finished successfully.")
                continue

            show_line = any(term in line_lower for term in progress_terms)
            show_line = show_line or (
                server_started and any(term in line_lower for term in error_terms)
            )
            status = friendly_status(line) if show_line else None
            if status:
                render_status(status)
except KeyboardInterrupt:
    process.terminate()
    raise
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
if return_code != 0:
    clear_output(wait=True)
    print("❌ Setup could not finish.")
    print("The latest technical details are shown below:\n")
    print("".join(recent_log_lines))
    print(f"\nComplete log: {LOG_PATH}")
    raise RuntimeError("ebook2audiobook failed to start; see the log above.")

render_status("👋 ebook2audiobook stopped. Run this cell again to restart it.")
